In [43]:
# PDF CHATBOT

import os
os.environ["GOOGLE_API_KEY"]=""

# Install Dependency
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-google-genai
!pip install -U faiss-cpu
!pip install -U pypdf

In [44]:
# Changing FAISS to Pinecone
!pip install -U pinecone langchain-pinecone

  Using cached pinecone-9.0.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)


In [45]:
os.environ["PINECONE_API_KEY"] = ""

In [46]:
# Import necessary libraries
from langchain_community.document_loaders import PyPDFLoader
# from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_core.output_parsers import StrOutputParser

In [47]:
# Adding Pinecone Import removing FAISS
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

In [48]:
# Initializing Pinecone Index
from pinecone import Pinecone
pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)
index = pc.Index("pdf-chatbot")

In [49]:
# Indexing Phase
# Step 1. Load PDF
loader = PyPDFLoader("Gagan_Bansal_CSE26.pdf")
document = loader.load()

In [50]:
# Step 2. Split Text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(document)

In [51]:
# Step 3. Create Vector Store
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)
# vector_store = FAISS.from_documents(chunks, embedding)
# Replacing FAISS to Pinecone 
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding,
    index_name="pdf-chatbot"
)

# RETRIVAL PHASE

In [53]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# Augmented Generation

In [55]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant. Use the following retrieved context to answer the question.
    If you don't know the answer, say you don't know.
    Context: {context}
    Question: {question}
    """,
    input_variables=["context", "question"],
)

In [56]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [57]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough(),
})

In [58]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [59]:
parser = StrOutputParser()

# Main Pipeline

In [60]:
main_chain = parallel_chain | prompt | llm | parser

In [61]:
response = main_chain.invoke("What is the main topic of the PDF?")

In [62]:
print(response)

The main topic of the PDF content is a summary of **software engineering projects and technical skills**. It details two specific projects, "oForge" and "KVMemo," along with experiences in debugging compatibility issues across different architectures.


In [63]:
response = main_chain.invoke("Can you summarize the PDF?")

In [64]:
print(response)

Gagan Bansal is a final-year Computer Science student with a CGPA of 9.41, set to graduate in June 2026. He has an R&D internship at Quark Software Inc. and practical experience in backend systems, applied AI development, and cross-platform C++ porting.

His technical skills encompass languages like C++, Java, and basic Python; web and backend technologies including React.js, TailwindCSS, Node.js, Express.js, Apache Kafka, and RESTful APIs; databases such as MongoDB, SQL, and PostgreSQL; and GenAI tools like LLM APIs, RAG Pipelines, Pinecone, Agentic AI, MCP, and LangChain. He is also proficient with tools and platforms like Figma, Git, Linux, macOS, and Claude Code.

As a Research and Development Intern at Quark Software Inc. (Jan 2026 – Present) in Chandigarh, he has successfully ported and modernized a legacy codebase to ensure compatibility with the latest macOS versions, supporting both ARM64 (Apple Silicon) and x86-64 architectures. He also resolved complex cross-platform build a